In [25]:
import pandas as pd
import re

In [26]:
df = pd.read_csv("IMDB Dataset.csv")
print(f"Dataset loaded: {df.shape[0]} rows, columns: {list(df.columns)}")
print(df.head())

Dataset loaded: 50000 rows, columns: ['review', 'sentiment']
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [27]:
# Clean reviews with regex

def clean_review(text):
    text = re.sub(r'<[^>]+>', ' ', text)        # strip HTML tags  e.g. <br />
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)    # keep only letters & spaces
    text = re.sub(r'\s+', ' ', text).strip()     # collapse multiple spaces
    return text.lower()

df['clean'] = df['review'].apply(clean_review)
print("Sample cleaned review:")
print(df['clean'].iloc[0][:200])

Sample cleaned review:
one of the other reviewers has mentioned that after watching just oz episode you ll be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its br


In [28]:
# Split into positive and negative subsets
positive_reviews = df[df['sentiment'] == 'positive']['clean']
negative_reviews = df[df['sentiment'] == 'negative']['clean']

total     = len(df)
total_pos = len(positive_reviews)
total_neg = len(negative_reviews)

print(f"Total reviews   : {total}")
print(f"Positive reviews: {total_pos}")
print(f"Negative reviews: {total_neg}")

# Prior probabilitY calculation
P_positive = total_pos / total
print(f"\nP(Positive) = {P_positive:.4f}")

Total reviews   : 50000
Positive reviews: 25000
Negative reviews: 25000

P(Positive) = 0.5000


In [29]:
# Helper to identify whether keyword appear as a whole word in the review
def contains_keyword(review_text, keyword):
    # \b is for making sure we match the whole word and not a substring
    pattern = r'\b' + keyword + r'\b'
    return bool(re.search(pattern, review_text))

In [30]:
# Bayesian computation for one keyword
# Returns four variables: prior, likelihood, marginal, posterior

def compute_bayes(keyword):

    # STEP 1: Count positive reviews that contain the keyword

    pos_with_kw = 0  # start counter at 0

    for review in positive_reviews:
        if contains_keyword(review, keyword):
            pos_with_kw = pos_with_kw + 1  # increase counter by 1


    # STEP 2: Count negative reviews that contain the keyword

    neg_with_kw = 0

    for review in negative_reviews:
        if contains_keyword(review, keyword):
            neg_with_kw = neg_with_kw + 1  # increase counter by 1


    # STEP 3: Count total reviews that contain the keyword

    total_with_kw = pos_with_kw + neg_with_kw


    # STEP 4: Get prior probability P(Positive)

    prior = P_positive


    # STEP 5: Calculate likelihood P(keyword | Positive)
    # i. e. Out of all positive reviews, how many contain the keyword?

    likelihood = pos_with_kw / total_pos

    # STEP 6: Calculate marginal probability P(keyword)
    # i.e. Out of ALL reviews, how many contain the keyword?

    marginal = total_with_kw / total

    # STEP 7: Apply Bayes' Theorem
    # P(Positive | keyword) =(P(keyword | Positive) * P(Positive)) / P(keyword)

    if marginal == 0:
        # Avoid division by zero
        posterior = 0.0
    else:
        posterior = (likelihood * prior) / marginal

    # STEP 8: Return all values
    return prior, likelihood, marginal, posterior


print("compute_bayes() defined and ready to run.")

compute_bayes() defined and ready to run.


We chose 2 positive and 2 negative keywords based on common language patterns in movie reviews:

*   **Positive sentiment**: `brilliant` and `amazing` which are typically used to praise acting, writing, or direction, rarely appearing in negative reviews.

*   **Negative sentiment:** `terrible` and `boring` which are usually used for low quality films and express loss of interest in the film.



In [31]:
# Selecting keywords

positive_keywords = ['brilliant', 'amazing']
negative_keywords = ['terrible', 'boring']
all_keywords      = positive_keywords + negative_keywords

# Choice of conditional probability
# We compute P(Positive | keyword) for all keywords.
# A high value means the keyword strongly indicates a positive review.
# A low value means the keyword strongly indicates a negative review.

print(f"{'Keyword':<12} {'Prior P(Pos)':>13} {'Likelihood P(kw|Pos)':>22} {'Marginal P(kw)':>16} {'Posterior P(Pos|kw)':>21}")
print("-" * 88)

for kw in all_keywords:
    prior, likelihood, marginal, posterior = compute_bayes(kw)
    print(f"{kw:<12} {prior:>13.4f} {likelihood:>22.4f} {marginal:>16.4f} {posterior:>21.4f}")

Keyword       Prior P(Pos)   Likelihood P(kw|Pos)   Marginal P(kw)   Posterior P(Pos|kw)
----------------------------------------------------------------------------------------
brilliant           0.5000                 0.0635           0.0418                0.7601
amazing             0.5000                 0.0672           0.0432                0.7780
terrible            0.5000                 0.0153           0.0540                0.1418
boring              0.5000                 0.0237           0.0610                0.1940


### Observations

- The **Prior P(Pos)** is 0.5000 for all keywords, indicating an equal chance of a review being positive or negative before considering any specific words.
- **Likelihood P(kw|Pos)** is the probability of a keyword appearing given that the review is positive. 'amazing' (0.0672) and 'brilliant' (0.0635) have higher likelihoods, meaning they are more common in positive reviews.
- The **Marginal P(kw)** tells us the overall frequency of each keyword in the entire dataset. 'boring' (0.0610) and 'terrible' (0.0540) appear more frequently than 'amazing' (0.0432) and 'brilliant' (0.0418).
- The **Posterior P(Pos|kw)** indicates the updated probability of a review being positive after observing the keyword:
    - Keywords like 'brilliant' (0.7601) and 'amazing' (0.7780) significantly increase the probability of a review being positive (from 0.5 to over 0.75), making them strong positive indicators.
    - On the other hand, 'terrible' (0.1418) and 'boring' (0.1940) significantly reduce the probability of a review being positive, confirming they are strong negative indicators.